# Proj2 — Light Full-Data 2-Config Sweep

This notebook is a low-memory version of the project sweep for DSMLP MIG.

**What it does:**
- Uses full `train.json` + `validation.json`
- Runs only **2 configs** (0.5B narrow and 1.5B narrow)
- Enables **QLoRA** to reduce VRAM pressure
- Uses conservative training args (`batch=1`, `max_length=512`, `num_chunks=1`)

| Catalog ID | Model | LR | LoRA |
|------------|-------|-----|------|
| 0 | Qwen2.5-0.5B-Instruct | 1e-4 | narrow (r=8) |
| 1 | Qwen2.5-1.5B-Instruct | 1e-4 | narrow (r=8) |

RapidFire assigns internal run ids `1..N` for selected configs.

## 0. Debug knobs (edit here)

In [1]:
from datetime import datetime

from paths import LOGS_DIR, REPO_ROOT, TRAIN_JSON, VAL_JSON, print_paths, require_release_data, setup

setup(chdir_try=True)  # cwd → try/; ./logs → try/logs; imports schema_*.py from here
require_release_data()
print_paths()

# --- What to run (catalog ids) ---
RUN_IDS = [0, 1]    # 0: 0.5B narrow, 1: 1.5B narrow

# --- Full-data but light training ---
NUM_EPOCHS = 2
MAX_LENGTH = 512
USE_QLORA = True

# --- Experiment naming (timestamp avoids RapidFire suffix churn) ---
EXPERIMENT_BASE = "proj2-debug-2config"
EXPERIMENT_NAME = f"{EXPERIMENT_BASE}-{datetime.now():%Y%m%d-%H%M}"
EXPERIMENT_PATH = "./logs"

# --- RapidFire scheduling ---
NUM_CHUNKS = 1      # reduce allocator fragmentation (worker reset per chunk)
NUM_GPUS = 1
SEED = 42

SCHEMA_STYLE = "minimal"         # or "full"

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"RUN_IDS={RUN_IDS}  epochs={NUM_EPOCHS}  max_length={MAX_LENGTH}  qlora={USE_QLORA}")
print(f"NUM_CHUNKS={NUM_CHUNKS}  (full train + val)")

REPO_ROOT   = /home/emc001/schema-linker-sft-pipeline
TRY_DIR     = /home/emc001/schema-linker-sft-pipeline/try
SCHEMAS_DIR = /home/emc001/schema-linker-sft-pipeline/schemas  (18 json files)
LOGS_DIR    = /home/emc001/schema-linker-sft-pipeline/try/logs
RF_HOME     = /home/emc001/rapidfireai
cwd         = /home/emc001/schema-linker-sft-pipeline/try
Experiment: proj2-debug-2config-20260528-1948
RUN_IDS=[0, 1]  epochs=2  max_length=512  qlora=True
NUM_CHUNKS=1  (full train + val)


## 1. Optional — free GPU / end stale experiment

In [3]:
import gc

def cuda_cleanup():
  """Call between failed runs to reduce allocator/NVML pressure."""
  gc.collect()
  try:
    import torch
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
      torch.cuda.synchronize()
      print("CUDA cache cleared")
  except Exception as e:
    print(f"cuda_cleanup skipped: {e}")

# Uncomment if a previous Experiment object is still in memory:
# experiment.end()
# cuda_cleanup()

## 2. Imports & data

In [4]:
import json

from datasets import Dataset
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

from schema_formatting import make_formatting_func
from schema_metrics import compute_metrics_for_rf

# TRAIN_JSON / VAL_JSON set in section 0 via paths.setup()
with open(TRAIN_JSON) as f:
    raw_train = json.load(f)
with open(VAL_JSON) as f:
    raw_val = json.load(f)

train_ds = Dataset.from_list(raw_train)
val_ds = Dataset.from_list(raw_val)
train_ds = train_ds.shuffle(seed=SEED)
fmt_func = make_formatting_func(SCHEMA_STYLE)

print(f"Train={len(train_ds)}  Val={len(val_ds)}  schema={SCHEMA_STYLE}")

Train=301  Val=101  schema=minimal


## 3. Model factory & config grid (2 lightweight runs)

In [5]:
def create_model(model_config: dict):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_name = model_config["model_name"]
    model_kwargs = model_config["model_kwargs"].copy()
    use_qlora = model_kwargs.pop("use_qlora", False)

    if use_qlora:
        from transformers import BitsAndBytesConfig
        import torch
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name, quantization_config=bnb_cfg, **model_kwargs
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    return (model, tokenizer)


peft_narrow = RFLoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
def make_sft_args(lr: float) -> RFSFTConfig:
    return RFSFTConfig(
        learning_rate=lr,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=NUM_EPOCHS,
        max_length=MAX_LENGTH,
        logging_steps=10,
        eval_strategy="epoch",
        load_best_model_at_end=False,
        gradient_checkpointing=True,
        bf16=True,
        fp16=False,
        report_to="none",
    )


def model_kwargs() -> dict:
    kw = {"device_map": "auto", "torch_dtype": "auto", "use_cache": False}
    if USE_QLORA:
        kw["use_qlora"] = True
    return kw


GEN_CFG = {"max_new_tokens": 128, "do_sample": False, "temperature": 1.0}

QWEN_05B = "Qwen/Qwen2.5-0.5B-Instruct"
QWEN_15B = "Qwen/Qwen2.5-1.5B-Instruct"

RUN_CATALOG = [
    {"run_id": 0, "model": QWEN_05B, "lr": 1e-4, "label": "0.5B | lr=1e-4 | narrow"},
    {"run_id": 1, "model": QWEN_15B, "lr": 1e-4, "label": "1.5B | lr=1e-4 | narrow"},
]


def catalog_entry_to_rf_config(entry: dict) -> RFModelConfig:
    return RFModelConfig(
        model_name=entry["model"],
        peft_config=peft_narrow,
        training_args=make_sft_args(entry["lr"]),
        model_type="causal_lm",
        model_kwargs=model_kwargs(),
        formatting_func=fmt_func,
        compute_metrics=compute_metrics_for_rf,
        generation_config=GEN_CFG.copy(),
    )


selected = [e for e in RUN_CATALOG if e["run_id"] in RUN_IDS]
if not selected:
    raise ValueError(f"RUN_IDS {RUN_IDS} matched no runs")

for i, e in enumerate(selected, start=1):
    print(f"  catalog {e['run_id']} -> run ~{i}: {e['label']}  qlora={USE_QLORA}")

config_set = List([catalog_entry_to_rf_config(e) for e in selected])
config_group = RFGridSearch(configs=config_set, trainer_type="SFT")

print(f"\nLaunching {len(selected)} run(s) on {len(train_ds)} train / {len(val_ds)} val examples")

  catalog 0 -> run ~1: 0.5B | lr=1e-4 | narrow  qlora=True
  catalog 1 -> run ~2: 1.5B | lr=1e-4 | narrow  qlora=True

Launching 2 run(s) on 301 train / 101 val examples


## 4. Create experiment & run

In [6]:
cuda_cleanup()

experiment = Experiment(
    experiment_name=EXPERIMENT_NAME,
    mode="fit",
    experiment_path=EXPERIMENT_PATH,
)
print(f"Logs: {EXPERIMENT_PATH}/{experiment.experiment_name}")

CUDA cache cleared
Experiment proj2-debug-2config-20260528-1948 created with Experiment ID: 2 and Metric Experiment ID: 2 at /home/emc001/schema-linker-sft-pipeline/try/logs/proj2-debug-2config-20260528-1948
Logs: ./logs/proj2-debug-2config-20260528-1948


In [7]:
# Full-data light run (2 configs). If OOM persists: lower MAX_LENGTH or NUM_EPOCHS in section 0.
experiment.run_fit(
    config_group,
    create_model,
    train_ds,
    val_ds,
    num_chunks=NUM_CHUNKS,
    seed=SEED,
    num_gpus=NUM_GPUS,
)

INFO 05-28 19:48:57 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-28 19:49:10 [__init__.py:216] Automatically detected platform cuda.


## 5. Results & cleanup

In [8]:
results = experiment.get_results()
display(results)

,run_id,step,loss,grad_norm,learning_rate,num_tokens,mean_token_accuracy,chunk number,num_epochs_completed,eval_loss,...,table_F1,col_P,col_R,col_F1,leaderboard_score,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,1,10,1.7912,0.0,0.000097,40960.0,0.016135,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,20,7.3047,0.0,0.000085,81920.0,0.087703,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,30,289.5419,0.0,0.000068,122880.0,0.055657,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1,38,NaN,NaN,NaN,154112.0,0.042069,NaN,NaN,NaN,...,0.0,0.0297,0.0297,0.0297,0.0149,396.5025,0.759,0.096,3.314383e+14,87.958759
3,1,40,0.0000,0.0,0.000048,8192.0,0.000000,0.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,50,13.2853,0.0,0.000028,49152.0,0.016135,0.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1,60,26.4195,0.0,0.000012,90112.0,0.095781,0.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1,70,219.5897,0.0,0.000002,131072.0,0.055766,0.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1,76,NaN,NaN,NaN,154112.0,0.042472,NaN,NaN,NaN,...,0.0,0.0297,0.0297,0.0297,0.0149,387.7056,0.784,0.098,1.660022e+14,137.309110
9,2,10,1.5606,0.0,0.000097,40960.0,0.016571,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
experiment.end()
cuda_cleanup()
print("Experiment ended.")

Experiment proj2-debug-2config-20260528-1948 ended
Workers stopped
CUDA cache cleared
Experiment ended.
